# 14. M1 Expanded Compact13 오분류 Audit

`expanded_compact13 + threshold 0.6` 기준에서 남은 FP/FN 6건을 해석하고, expanded dataset을 다음 기준으로 잠글 수 있는지 확인한다.

고정 기준:
- fixed eval: `strict_no_event20` 49행
- feature: `compact13`
- model: `LogisticRegression(class_weight="balanced")`
- threshold: `0.6`
- normal 라벨은 변경하지 않음


In [1]:
from __future__ import annotations

from pathlib import Path
import zipfile
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 160

DATA_DIR = Path("05_데이터셋") / "PreDist"
ZIP_PATH = DATA_DIR / "predist_dataset.zip"
NOTEBOOK_PATH = Path("06_노트북") / "14_m1_expanded_compact13_error_audit.ipynb"
OUTPUT_DIR = Path("07_데이터산출물")

FEATURE_POOL_PATH = OUTPUT_DIR / "m1_expansion_feature_pool.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "m1_expanded_training_fixed_eval_predictions.csv"
HARD_NORMAL_REVIEW_PATH = OUTPUT_DIR / "m1_hard_normal_timeseries_review.csv"

ERROR_AUDIT_PATH = OUTPUT_DIR / "m1_expanded_compact13_error_audit.csv"
ERROR_FEATURE_PROFILE_PATH = OUTPUT_DIR / "m1_expanded_compact13_error_feature_profile.csv"
TRANSITION_AUDIT_PATH = OUTPUT_DIR / "m1_expanded_compact13_transition_audit.csv"
CANDIDATE_ABLATION_METRICS_PATH = OUTPUT_DIR / "m1_expanded_compact13_candidate_ablation_metrics.csv"
FINAL_LOCK_DECISION_PATH = OUTPUT_DIR / "m1_final_dataset_lock_decision.csv"
REPORT_PATH = OUTPUT_DIR / "14_M1_expanded_compact13_error_audit_보고서.md"

PROBABILITY_PNG = OUTPUT_DIR / "m1_expanded_compact13_error_probability_ranking.png"
FEATURE_HEATMAP_PNG = OUTPUT_DIR / "m1_expanded_compact13_error_feature_heatmap.png"
TRANSITION_SUMMARY_PNG = OUTPUT_DIR / "m1_expanded_compact13_transition_summary.png"

SOURCE_PREFIX = "manufacturer 1"
THRESHOLD = 0.6
RANDOM_STATE = 42
EXCLUDED_EVENT_IDS = {20, 34, 69}
EXPECTED_FP_EVENTS = {27, 28, 68}
EXPECTED_FN_EVENTS = {15, 40, 52}


## 1. 입력 로드와 기준 확인


In [2]:
feature_pool = pd.read_csv(FEATURE_POOL_PATH)
predictions = pd.read_csv(PREDICTIONS_PATH)
hard_normal_review = pd.read_csv(HARD_NORMAL_REVIEW_PATH)

for frame in [feature_pool, predictions, hard_normal_review]:
    if "source_event_id" in frame.columns:
        frame["source_event_id"] = pd.to_numeric(frame["source_event_id"], errors="coerce")
    if "substation_id" in frame.columns:
        frame["substation_id"] = frame["substation_id"].astype(int)
    for col in ["window_start", "window_end"]:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")

COMPACT13_FEATURES = [col for col in feature_pool.columns if "__" in col]
fixed_eval = feature_pool.loc[feature_pool["pool_role"].eq("fixed_eval")].copy()
candidate_pool = feature_pool.loc[feature_pool["pool_role"].eq("expansion_candidate")].copy()
fixed_eval["source_event_id"] = fixed_eval["source_event_id"].astype(int)
predictions["source_event_id"] = predictions["source_event_id"].astype(int)

assert len(COMPACT13_FEATURES) == 13
assert len(fixed_eval) == 49
assert fixed_eval["label"].value_counts().to_dict() == {"normal": 35, "efd_possible": 14}
assert not feature_pool["source_event_id"].dropna().astype(int).isin(EXCLUDED_EVENT_IDS).any()

fixed_eval[["sample_id", "source_event_id", "substation_id", "label", "review_tag"]].head()


,sample_id,source_event_id,substation_id,label,review_tag
0,normal_0002,2,9,normal,NaN
1,normal_0004,4,15,normal,NaN
2,normal_0008,8,6,normal,hard_normal_metadata
3,normal_0012,12,19,normal,NaN
4,normal_0014,14,26,normal,NaN


## 2. FP/FN 오분류 추출


In [3]:
expanded_predictions = predictions.loc[
    predictions["strategy"].eq("expanded_compact13")
    & predictions["threshold"].eq(THRESHOLD)
].copy()
reference_predictions = predictions.loc[
    predictions["strategy"].eq("reference_compact13")
    & predictions["threshold"].eq(THRESHOLD)
].copy()


def classify_error_type(y_true: int, y_pred: int) -> str:
    if int(y_true) == 0 and int(y_pred) == 1:
        return "FP"
    if int(y_true) == 1 and int(y_pred) == 0:
        return "FN"
    return "OK"


expanded_predictions["error_type"] = [
    classify_error_type(y_true, y_pred)
    for y_true, y_pred in zip(expanded_predictions["y_true"], expanded_predictions["y_pred"])
]
reference_predictions["error_type"] = [
    classify_error_type(y_true, y_pred)
    for y_true, y_pred in zip(reference_predictions["y_true"], reference_predictions["y_pred"])
]

expanded_errors = expanded_predictions.loc[
    expanded_predictions["error_type"].isin(["FP", "FN"])
].copy()

assert set(expanded_errors.loc[expanded_errors["error_type"].eq("FP"), "source_event_id"]) == EXPECTED_FP_EVENTS
assert set(expanded_errors.loc[expanded_errors["error_type"].eq("FN"), "source_event_id"]) == EXPECTED_FN_EVENTS
assert len(expanded_errors) == 6

expanded_errors[[
    "error_type",
    "source_event_id",
    "substation_id",
    "label",
    "review_tag",
    "y_probability",
    "prediction_label",
    "coverage_rate",
    "fold",
]].sort_values(["error_type", "source_event_id"])


,error_type,source_event_id,substation_id,label,review_tag,y_probability,prediction_label,coverage_rate,fold
118,FN,15,29,efd_possible,NaN,0.218518,normal,1.00000,4
146,FN,40,24,efd_possible,NaN,0.302124,normal,0.99504,5
119,FN,52,21,efd_possible,NaN,0.564281,normal,1.00000,4
140,FP,27,12,normal,hard_normal_metadata,0.982435,efd_possible,1.00000,5
85,FP,28,7,normal,NaN,0.965007,efd_possible,1.00000,3
29,FP,68,13,normal,review_required_normal,0.626928,efd_possible,1.00000,1


## 3. disturbance/fault overlap과 label 품질 보강


In [4]:
def read_m1_csv(name: str) -> pd.DataFrame:
    with zipfile.ZipFile(ZIP_PATH) as zf:
        with zf.open(f"{SOURCE_PREFIX}/{name}") as handle:
            return pd.read_csv(handle, sep=";")


faults = read_m1_csv("faults.csv")
disturbances = read_m1_csv("disturbances.csv")
faults["Event ID"] = faults["Event ID"].astype(int)
faults["substation ID"] = faults["substation ID"].astype(int)
faults["Report date"] = pd.to_datetime(faults["Report date"], errors="coerce")
disturbances["substation ID"] = disturbances["substation ID"].astype(int)
disturbances["Event start"] = pd.to_datetime(disturbances["Event start"], errors="coerce")

fault_windows = []
for _, fault in faults.dropna(subset=["Report date"]).iterrows():
    end = pd.Timestamp(fault["Report date"])
    fault_windows.append(
        {
            "substation_id": int(fault["substation ID"]),
            "window_start": end - pd.Timedelta(days=7),
            "window_end": end,
        }
    )
fault_windows = pd.DataFrame(fault_windows)


def disturbance_count_in_window(substation_id: int, start, end) -> int:
    same = disturbances.loc[disturbances["substation ID"].eq(int(substation_id))]
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    return int(((same["Event start"] >= start) & (same["Event start"] < end)).sum())


def fault_overlap_count(substation_id: int, start, end) -> int:
    same = fault_windows.loc[fault_windows["substation_id"].eq(int(substation_id))]
    if same.empty:
        return 0
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    return int(((same["window_start"] < end) & (same["window_end"] > start)).sum())


hard_review_map = hard_normal_review.set_index("source_event_id")
metadata_cols = [
    "candidate_type",
    "label_strength",
    "window_start",
    "window_end",
    "window_policy",
    "sample_count",
    "expected_sample_count",
    "source_type",
    "fault_label",
] + COMPACT13_FEATURES
error_audit = expanded_errors.merge(
    fixed_eval[["source_event_id"] + metadata_cols],
    on="source_event_id",
    how="left",
)

error_rows = []
for _, row in error_audit.iterrows():
    event_id = int(row["source_event_id"])
    disturbance_count = disturbance_count_in_window(
        int(row["substation_id"]), row["window_start"], row["window_end"]
    )
    overlap_count = fault_overlap_count(
        int(row["substation_id"]), row["window_start"], row["window_end"]
    )
    hard_review_label = ""
    hard_review_reason = ""
    if event_id in hard_review_map.index:
        hard_review_label = str(hard_review_map.loc[event_id, "review_label"])
        hard_review_reason = str(hard_review_map.loc[event_id, "review_reason"])
    fault_label = "" if pd.isna(row["fault_label"]) else str(row["fault_label"])
    label_quality_issue = bool(
        row["error_type"] == "FN"
        and (float(row["coverage_rate"]) < 0.95 or fault_label.strip() == "" or fault_label.lower() == "unknown")
    )
    new_hard_normal_candidate = bool(
        row["error_type"] == "FP"
        and event_id == 28
        and disturbance_count == 0
        and overlap_count == 0
    )
    if row["error_type"] == "FP":
        audit_note = "normal sample predicted as efd_possible"
        if hard_review_label:
            audit_note += f"; existing tag={hard_review_label}"
        if new_hard_normal_candidate:
            audit_note += "; new hard normal candidate without overlap"
    else:
        audit_note = "efd_possible sample predicted as normal"
        audit_note += "; label quality check passed" if not label_quality_issue else "; label quality issue"
    error_rows.append(
        {
            **row.to_dict(),
            "disturbance_count": disturbance_count,
            "fault_window_overlap_count": overlap_count,
            "hard_normal_review_label": hard_review_label,
            "hard_normal_review_reason": hard_review_reason,
            "new_hard_normal_candidate": new_hard_normal_candidate,
            "label_quality_issue": label_quality_issue,
            "audit_note": audit_note,
        }
    )

error_audit = pd.DataFrame(error_rows)
error_audit[[
    "error_type",
    "source_event_id",
    "substation_id",
    "label",
    "review_tag",
    "hard_normal_review_label",
    "disturbance_count",
    "fault_window_overlap_count",
    "label_quality_issue",
    "new_hard_normal_candidate",
    "audit_note",
]].sort_values(["error_type", "source_event_id"])


,error_type,source_event_id,substation_id,label,review_tag,hard_normal_review_label,disturbance_count,fault_window_overlap_count,label_quality_issue,new_hard_normal_candidate,audit_note
2,FN,15,29,efd_possible,NaN,,0,1,False,False,efd_possible sample predicted as normal; label...
5,FN,40,24,efd_possible,NaN,,3,1,False,False,efd_possible sample predicted as normal; label...
3,FN,52,21,efd_possible,NaN,,1,2,False,False,efd_possible sample predicted as normal; label...
4,FP,27,12,normal,hard_normal_metadata,hard_normal_metadata,0,0,False,False,normal sample predicted as efd_possible; exist...
1,FP,28,7,normal,NaN,,0,0,False,True,normal sample predicted as efd_possible; new h...
0,FP,68,13,normal,review_required_normal,review_required_normal,0,0,False,False,normal sample predicted as efd_possible; exist...


## 4. reference 대비 transition audit


In [5]:
transition = reference_predictions[
    [
        "sample_id",
        "source_event_id",
        "substation_id",
        "label",
        "review_tag",
        "y_true",
        "fold",
        "y_probability",
        "y_pred",
        "prediction_label",
        "error_type",
    ]
].rename(
    columns={
        "y_probability": "reference_probability",
        "y_pred": "reference_pred",
        "prediction_label": "reference_prediction_label",
        "error_type": "reference_error_type",
    }
).merge(
    expanded_predictions[
        [
            "sample_id",
            "source_event_id",
            "y_probability",
            "y_pred",
            "prediction_label",
            "error_type",
        ]
    ].rename(
        columns={
            "y_probability": "expanded_probability",
            "y_pred": "expanded_pred",
            "prediction_label": "expanded_prediction_label",
            "error_type": "expanded_error_type",
        }
    ),
    on=["sample_id", "source_event_id"],
    how="inner",
)


def classify_transition(reference_error: str, expanded_error: str) -> str:
    reference_is_error = reference_error in {"FP", "FN"}
    expanded_is_error = expanded_error in {"FP", "FN"}
    if reference_is_error and not expanded_is_error:
        return "corrected_error"
    if not reference_is_error and expanded_is_error:
        return "new_error"
    if reference_is_error and expanded_is_error:
        return "unchanged_error"
    return "unchanged_correct"


transition["transition_type"] = [
    classify_transition(ref, exp)
    for ref, exp in zip(transition["reference_error_type"], transition["expanded_error_type"])
]
transition["probability_delta"] = transition["expanded_probability"] - transition["reference_probability"]
transition = transition.merge(
    fixed_eval[["source_event_id", "window_start", "window_end", "coverage_rate", "fault_label"]],
    on="source_event_id",
    how="left",
)
transition.to_csv(TRANSITION_AUDIT_PATH, index=False, encoding="utf-8-sig")

transition.loc[
    transition["source_event_id"].isin(sorted(EXPECTED_FP_EVENTS | EXPECTED_FN_EVENTS)),
    [
        "source_event_id",
        "label",
        "reference_error_type",
        "expanded_error_type",
        "transition_type",
        "reference_probability",
        "expanded_probability",
        "probability_delta",
    ],
].sort_values("source_event_id")


,source_event_id,label,reference_error_type,expanded_error_type,transition_type,reference_probability,expanded_probability,probability_delta
38,15,efd_possible,FN,FN,unchanged_error,0.027698,0.218518,0.190820
42,27,normal,FP,FP,unchanged_error,0.671341,0.982435,0.311094
23,28,normal,OK,FP,new_error,0.509508,0.965007,0.455498
48,40,efd_possible,FN,FN,unchanged_error,0.053631,0.302124,0.248493
39,52,efd_possible,FN,FN,unchanged_error,0.120825,0.564281,0.443456
7,68,normal,FP,FP,unchanged_error,0.719254,0.626928,-0.092327


## 5. fold별 expanded 모델 재학습과 contribution 계산


In [6]:
def make_logistic_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                    max_iter=1000,
                ),
            ),
        ]
    )


def probability_for_class_one(model: Pipeline, X: pd.DataFrame) -> np.ndarray:
    probabilities = model.predict_proba(X)
    if 1 not in model.classes_:
        return np.zeros(len(X))
    class_index = list(model.classes_).index(1)
    return probabilities[:, class_index]


def short_feature_name(feature: str) -> str:
    return (
        feature.replace("s_hc1_supply_temperature_setpoint", "supply_setpoint")
        .replace("s_hc1_supply_temperature_error", "supply_error")
        .replace("s_hc1_supply_temperature", "supply_temp")
        .replace("p_hc1_return_temperature", "hc1_return")
        .replace("p_net_return_temperature", "net_return")
        .replace("p_net_meter_flow", "flow")
        .replace("outdoor_temperature", "outdoor")
        .replace("p_return_gap", "return_gap")
        .replace("__", " / ")
        .replace("_minus_", "-")
        .replace("_", " ")
    )


sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
profile_rows = []
fold_group_audit = []
fixed_eval_ordered = fixed_eval.reset_index(drop=True).copy()

for fold, (train_idx, test_idx) in enumerate(
    sgkf.split(
        fixed_eval_ordered[COMPACT13_FEATURES],
        fixed_eval_ordered["y"],
        fixed_eval_ordered["substation_id"],
    ),
    start=1,
):
    fixed_train = fixed_eval_ordered.iloc[train_idx].copy()
    fixed_test = fixed_eval_ordered.iloc[test_idx].copy()
    test_groups = set(fixed_test["substation_id"].astype(int))
    train_groups = set(fixed_train["substation_id"].astype(int))
    candidate_train = candidate_pool.loc[
        ~candidate_pool["substation_id"].astype(int).isin(test_groups)
    ].copy()
    assert train_groups.isdisjoint(test_groups)
    assert set(candidate_train["substation_id"].astype(int)).isdisjoint(test_groups)
    expected_test_ids = set(
        expanded_predictions.loc[expanded_predictions["fold"].eq(fold), "source_event_id"]
    )
    assert expected_test_ids == set(fixed_test["source_event_id"].astype(int))

    train_rows = pd.concat([fixed_train, candidate_train], ignore_index=True)
    model = make_logistic_pipeline()
    model.fit(train_rows[COMPACT13_FEATURES], train_rows["y"].astype(int))

    test_errors = fixed_test.loc[
        fixed_test["source_event_id"].astype(int).isin(
            expanded_errors["source_event_id"].astype(int)
        )
    ].copy()
    if test_errors.empty:
        fold_group_audit.append(
            {
                "fold": fold,
                "group_overlap_count": len(train_groups.intersection(test_groups)),
                "test_groups": "|".join(str(v) for v in sorted(test_groups)),
            }
        )
        continue
    transformed = model.named_steps["scaler"].transform(
        model.named_steps["imputer"].transform(test_errors[COMPACT13_FEATURES])
    )
    coefficients = model.named_steps["model"].coef_[0]

    for row_index, (_, sample_row) in enumerate(test_errors.iterrows()):
        event_id = int(sample_row["source_event_id"])
        expanded_row = expanded_predictions.loc[
            expanded_predictions["source_event_id"].eq(event_id)
        ].iloc[0]
        for feature_index, feature in enumerate(COMPACT13_FEATURES):
            standardized_value = float(transformed[row_index, feature_index])
            coefficient = float(coefficients[feature_index])
            contribution = standardized_value * coefficient
            profile_rows.append(
                {
                    "source_event_id": event_id,
                    "sample_id": sample_row["sample_id"],
                    "substation_id": int(sample_row["substation_id"]),
                    "label": sample_row["label"],
                    "error_type": expanded_row["error_type"],
                    "fold": fold,
                    "feature": feature,
                    "feature_short": short_feature_name(feature),
                    "raw_value": float(sample_row[feature]),
                    "standardized_value": standardized_value,
                    "coefficient": coefficient,
                    "contribution": contribution,
                    "abs_contribution": abs(contribution),
                    "driving_direction": "positive_drive" if contribution >= 0 else "normal_drive",
                }
            )
    fold_group_audit.append(
        {
            "fold": fold,
            "group_overlap_count": len(train_groups.intersection(test_groups)),
            "test_groups": "|".join(str(v) for v in sorted(test_groups)),
        }
    )

feature_profile = pd.DataFrame(profile_rows)
feature_profile["contribution_rank_abs"] = feature_profile.groupby("source_event_id")[
    "abs_contribution"
].rank(method="first", ascending=False)
feature_profile["contribution_rank_direction"] = feature_profile.groupby(
    ["source_event_id", "driving_direction"]
)["abs_contribution"].rank(method="first", ascending=False)
feature_profile.to_csv(ERROR_FEATURE_PROFILE_PATH, index=False, encoding="utf-8-sig")

fold_group_audit = pd.DataFrame(fold_group_audit)
feature_profile.head()


,source_event_id,sample_id,substation_id,label,error_type,fold,feature,feature_short,raw_value,standardized_value,coefficient,contribution,abs_contribution,driving_direction,contribution_rank_abs,contribution_rank_direction
0,68,normal_0068,13,normal,FP,1,outdoor_temperature__last_12h_mean_minus_prev_...,outdoor / last 12h mean-prev 12h mean,-2.606944,-1.592435,-1.334509,2.125119,2.125119,positive_drive,1.0,1.0
1,68,normal_0068,13,normal,FP,1,outdoor_temperature__last_6h_mean_minus_prev_6...,outdoor / last 6h mean-prev 6h mean,-2.346111,-0.704663,1.874548,-1.320926,1.320926,normal_drive,2.0,1.0
2,68,normal_0068,13,normal,FP,1,outdoor_temperature__last_minus_first,outdoor / last-first,3.220000,0.297850,-0.070101,-0.020880,0.020880,normal_drive,11.0,4.0
3,68,normal_0068,13,normal,FP,1,p_hc1_return_temperature__last_12h_mean_minus_...,hc1 return / last 12h mean-prev 12h mean,2.740556,1.020423,-0.485157,-0.495065,0.495065,normal_drive,4.0,2.0
4,68,normal_0068,13,normal,FP,1,p_hc1_return_temperature__last_1d_mean_minus_p...,hc1 return / last 1d mean-prev 6d mean,-3.149954,-0.821533,-0.535176,0.439665,0.439665,positive_drive,5.0,3.0


## 6. error audit CSV 완성


In [7]:
def top_features_for_event(event_id: int, direction: str, top_n: int = 3) -> str:
    subset = feature_profile.loc[
        feature_profile["source_event_id"].eq(int(event_id))
        & feature_profile["driving_direction"].eq(direction)
    ].sort_values("abs_contribution", ascending=False)
    return "|".join(subset.head(top_n)["feature_short"].tolist())


error_audit["top_positive_driving_features"] = [
    top_features_for_event(event_id, "positive_drive")
    for event_id in error_audit["source_event_id"]
]
error_audit["top_normal_driving_features"] = [
    top_features_for_event(event_id, "normal_drive")
    for event_id in error_audit["source_event_id"]
]

transition_lookup = transition.set_index("source_event_id")
error_audit["reference_error_type"] = [
    transition_lookup.loc[int(event_id), "reference_error_type"]
    for event_id in error_audit["source_event_id"]
]
error_audit["transition_type"] = [
    transition_lookup.loc[int(event_id), "transition_type"]
    for event_id in error_audit["source_event_id"]
]
error_audit["reference_probability"] = [
    transition_lookup.loc[int(event_id), "reference_probability"]
    for event_id in error_audit["source_event_id"]
]
error_audit["probability_delta_vs_reference"] = [
    transition_lookup.loc[int(event_id), "probability_delta"]
    for event_id in error_audit["source_event_id"]
]

audit_columns = [
    "error_type",
    "source_event_id",
    "sample_id",
    "substation_id",
    "label",
    "label_strength",
    "review_tag",
    "hard_normal_review_label",
    "fold",
    "y_probability",
    "prediction_label",
    "reference_probability",
    "probability_delta_vs_reference",
    "reference_error_type",
    "transition_type",
    "coverage_rate",
    "sample_count",
    "expected_sample_count",
    "window_start",
    "window_end",
    "window_policy",
    "fault_label",
    "disturbance_count",
    "fault_window_overlap_count",
    "new_hard_normal_candidate",
    "label_quality_issue",
    "top_positive_driving_features",
    "top_normal_driving_features",
    "audit_note",
]
error_audit[audit_columns].sort_values(["error_type", "source_event_id"]).to_csv(
    ERROR_AUDIT_PATH, index=False, encoding="utf-8-sig"
)
error_audit[audit_columns].sort_values(["error_type", "source_event_id"])


,error_type,source_event_id,sample_id,substation_id,label,label_strength,review_tag,hard_normal_review_label,fold,y_probability,...,window_end,window_policy,fault_label,disturbance_count,fault_window_overlap_count,new_hard_normal_candidate,label_quality_issue,top_positive_driving_features,top_normal_driving_features,audit_note
2,FN,15,strict_positive_0015,29,efd_possible,strict_positive,NaN,,4,0.218518,...,2020-03-07 07:59:43,report_pre_7d,Failure of the domestic hot water storage char...,0,1,False,False,supply error / last-first|outdoor / last 12h m...,net return / last 1d mean-prev 6d mean|hc1 ret...,efd_possible sample predicted as normal; label...
5,FN,40,strict_positive_0040,24,efd_possible,strict_positive,NaN,,5,0.302124,...,2016-04-07 07:47:00,report_pre_7d,"Safety relief valve: Water loss, does not clos...",3,1,False,False,outdoor / last 12h mean-prev 12h mean|hc1 retu...,hc1 return / last 12h mean-prev 12h mean|suppl...,efd_possible sample predicted as normal; label...
3,FN,52,strict_positive_0052,21,efd_possible,strict_positive,NaN,,4,0.564281,...,2016-12-12 15:55:00,report_pre_7d,Control unit: Incorrect parameterisation,1,2,False,False,outdoor / last 6h mean-prev 6h mean|outdoor / ...,hc1 return / last 1d mean-prev 6d mean|net ret...,efd_possible sample predicted as normal; label...
4,FP,27,normal_0027,12,normal,normal,hard_normal_metadata,hard_normal_metadata,5,0.982435,...,2015-07-01 13:34:00,normal_event_7d,NaN,0,0,False,False,outdoor / last 6h mean-prev 6h mean|outdoor / ...,return gap / last 1d mean-prev 6d mean|return ...,normal sample predicted as efd_possible; exist...
1,FP,28,normal_0028,7,normal,normal,NaN,,3,0.965007,...,2019-01-08 10:29:00,normal_event_7d,NaN,0,0,True,False,outdoor / last 6h mean-prev 6h mean|outdoor / ...,hc1 return / last 1d mean-prev 6d mean|net ret...,normal sample predicted as efd_possible; new h...
0,FP,68,normal_0068,13,normal,normal,review_required_normal,review_required_normal,1,0.626928,...,2019-12-27 00:00:00,normal_event_7d,NaN,0,0,False,False,outdoor / last 12h mean-prev 12h mean|flow / l...,outdoor / last 6h mean-prev 6h mean|hc1 return...,normal sample predicted as efd_possible; exist...


## 7. candidate ablation


In [8]:
def metric_row(data: pd.DataFrame, strategy: str, fold: str | int, candidate_train_rows: int) -> dict:
    y_true = data["y_true"].astype(int)
    y_pred = data["y_pred"].astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "strategy": strategy,
        "fold": fold,
        "n": int(len(data)),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "false_positive_count": int(fp),
        "false_negative_count": int(fn),
        "true_positive_count": int(tp),
        "true_negative_count": int(tn),
        "false_positive_rate": fp / (fp + tn) if (fp + tn) else 0.0,
        "hard_normal_35_48_fp_count": int(
            data.loc[data["source_event_id"].isin([35, 48]), "y_pred"].sum()
        ),
        "review_required_19_68_fp_count": int(
            data.loc[data["source_event_id"].isin([19, 68]), "y_pred"].sum()
        ),
        "candidate_train_rows": candidate_train_rows,
    }


ablation_prediction_rows = []
ablation_fold_candidate_counts = {}

for fold, (train_idx, test_idx) in enumerate(
    sgkf.split(
        fixed_eval_ordered[COMPACT13_FEATURES],
        fixed_eval_ordered["y"],
        fixed_eval_ordered["substation_id"],
    ),
    start=1,
):
    fixed_train = fixed_eval_ordered.iloc[train_idx].copy()
    fixed_test = fixed_eval_ordered.iloc[test_idx].copy()
    test_groups = set(fixed_test["substation_id"].astype(int))
    base_candidate_train = candidate_pool.loc[
        ~candidate_pool["substation_id"].astype(int).isin(test_groups)
    ].copy()
    candidate_sets = {
        "reference": base_candidate_train.iloc[0:0].copy(),
        "weak_only": base_candidate_train.loc[base_candidate_train["candidate_type"].eq("weak_positive")],
        "normal_only": base_candidate_train.loc[base_candidate_train["candidate_type"].eq("candidate_normal")],
        "expanded_both": base_candidate_train,
    }
    for strategy, candidate_train in candidate_sets.items():
        train_rows = pd.concat([fixed_train, candidate_train], ignore_index=True)
        model = make_logistic_pipeline()
        model.fit(train_rows[COMPACT13_FEATURES], train_rows["y"].astype(int))
        probabilities = probability_for_class_one(model, fixed_test[COMPACT13_FEATURES])
        preds = (probabilities >= THRESHOLD).astype(int)
        ablation_fold_candidate_counts[(strategy, fold)] = len(candidate_train)
        for row, probability, pred in zip(fixed_test.itertuples(index=False), probabilities, preds):
            ablation_prediction_rows.append(
                {
                    "strategy": strategy,
                    "fold": fold,
                    "source_event_id": int(row.source_event_id),
                    "substation_id": int(row.substation_id),
                    "label": row.label,
                    "review_tag": row.review_tag,
                    "y_true": int(row.y),
                    "y_probability": float(probability),
                    "y_pred": int(pred),
                    "prediction_label": "efd_possible" if int(pred) == 1 else "normal",
                }
            )

ablation_predictions = pd.DataFrame(ablation_prediction_rows)
ablation_metric_rows = []
for strategy, strategy_data in ablation_predictions.groupby("strategy"):
    candidate_rows_all = int(np.mean([
        count for (s, _), count in ablation_fold_candidate_counts.items() if s == strategy
    ]))
    ablation_metric_rows.append(metric_row(strategy_data, strategy, "all", candidate_rows_all))
    for fold, fold_data in strategy_data.groupby("fold"):
        ablation_metric_rows.append(
            metric_row(
                fold_data,
                strategy,
                int(fold),
                ablation_fold_candidate_counts[(strategy, int(fold))],
            )
        )

ablation_metrics = pd.DataFrame(ablation_metric_rows).sort_values(["strategy", "fold"])
ablation_metrics.to_csv(CANDIDATE_ABLATION_METRICS_PATH, index=False, encoding="utf-8-sig")
ablation_metrics.loc[ablation_metrics["fold"].astype(str).eq("all")].sort_values("strategy")


,strategy,fold,n,balanced_accuracy,precision,recall,f1,false_positive_count,false_negative_count,true_positive_count,true_negative_count,false_positive_rate,hard_normal_35_48_fp_count,review_required_19_68_fp_count,candidate_train_rows
0,expanded_both,all,49,0.850000,0.785714,0.785714,0.785714,3,3,11,32,0.085714,0,1,68
6,normal_only,all,49,0.850000,0.785714,0.785714,0.785714,3,3,11,32,0.085714,0,1,58
12,reference,all,49,0.828571,0.833333,0.714286,0.769231,2,4,10,33,0.057143,0,1,0
18,weak_only,all,49,0.850000,0.785714,0.785714,0.785714,3,3,11,32,0.085714,0,1,10


## 8. 최종 dataset lock decision


In [9]:
def ablation_all(strategy: str) -> pd.Series:
    row = ablation_metrics.loc[
        ablation_metrics["strategy"].eq(strategy)
        & ablation_metrics["fold"].astype(str).eq("all")
    ]
    assert len(row) == 1
    return row.iloc[0]


reference_all = ablation_all("reference")
expanded_all = ablation_all("expanded_both")

event28 = error_audit.loc[error_audit["source_event_id"].eq(28)].iloc[0]
fn_errors = error_audit.loc[error_audit["error_type"].eq("FN")]

decision_rows = [
    {
        "criterion": "balanced_accuracy_no_drop",
        "reference_value": float(reference_all["balanced_accuracy"]),
        "expanded_value": float(expanded_all["balanced_accuracy"]),
        "delta": float(expanded_all["balanced_accuracy"] - reference_all["balanced_accuracy"]),
        "pass": bool(expanded_all["balanced_accuracy"] >= reference_all["balanced_accuracy"]),
        "note": "expanded_both fixed eval balanced accuracy is not lower than reference",
    },
    {
        "criterion": "recall_no_drop",
        "reference_value": float(reference_all["recall"]),
        "expanded_value": float(expanded_all["recall"]),
        "delta": float(expanded_all["recall"] - reference_all["recall"]),
        "pass": bool(expanded_all["recall"] >= reference_all["recall"]),
        "note": "positive recall is not lower than reference",
    },
    {
        "criterion": "fpr_delta_within_0_05",
        "reference_value": float(reference_all["false_positive_rate"]),
        "expanded_value": float(expanded_all["false_positive_rate"]),
        "delta": float(expanded_all["false_positive_rate"] - reference_all["false_positive_rate"]),
        "pass": bool((expanded_all["false_positive_rate"] - reference_all["false_positive_rate"]) <= 0.05),
        "note": "false positive rate increase stays within 0.05",
    },
    {
        "criterion": "hard_normal_35_48_not_fp",
        "reference_value": float(reference_all["hard_normal_35_48_fp_count"]),
        "expanded_value": float(expanded_all["hard_normal_35_48_fp_count"]),
        "delta": float(expanded_all["hard_normal_35_48_fp_count"] - reference_all["hard_normal_35_48_fp_count"]),
        "pass": bool(expanded_all["hard_normal_35_48_fp_count"] == 0),
        "note": "Event 35/48 did not become FP",
    },
    {
        "criterion": "new_fp_event28_interpretable",
        "reference_value": 0.0,
        "expanded_value": float(event28["new_hard_normal_candidate"]),
        "delta": float(event28["new_hard_normal_candidate"]),
        "pass": bool(event28["new_hard_normal_candidate"]),
        "note": "Event 28 has no disturbance/fault overlap and is kept as hard normal candidate",
    },
    {
        "criterion": "fn_label_quality_passed",
        "reference_value": 0.0,
        "expanded_value": float(fn_errors["label_quality_issue"].sum()),
        "delta": float(fn_errors["label_quality_issue"].sum()),
        "pass": bool(fn_errors["label_quality_issue"].sum() == 0),
        "note": "FN events 15/40/52 have sufficient coverage and known fault labels",
    },
]
final_decision = (
    "expanded_compact13_dataset_lock"
    if all(row["pass"] for row in decision_rows)
    else "candidate_selection_review_required"
)
decision_df = pd.DataFrame(decision_rows)
decision_df["final_decision"] = final_decision
decision_df.to_csv(FINAL_LOCK_DECISION_PATH, index=False, encoding="utf-8-sig")
decision_df


,criterion,reference_value,expanded_value,delta,pass,note,final_decision
0,balanced_accuracy_no_drop,0.828571,0.850000,0.021429,True,expanded_both fixed eval balanced accuracy is ...,expanded_compact13_dataset_lock
1,recall_no_drop,0.714286,0.785714,0.071429,True,positive recall is not lower than reference,expanded_compact13_dataset_lock
2,fpr_delta_within_0_05,0.057143,0.085714,0.028571,True,false positive rate increase stays within 0.05,expanded_compact13_dataset_lock
3,hard_normal_35_48_not_fp,0.000000,0.000000,0.000000,True,Event 35/48 did not become FP,expanded_compact13_dataset_lock
4,new_fp_event28_interpretable,0.000000,1.000000,1.000000,True,Event 28 has no disturbance/fault overlap and ...,expanded_compact13_dataset_lock
5,fn_label_quality_passed,0.000000,0.000000,0.000000,True,FN events 15/40/52 have sufficient coverage an...,expanded_compact13_dataset_lock


## 9. 시각화


In [10]:
error_plot = error_audit[audit_columns].sort_values("y_probability", ascending=True).copy()
colors = error_plot["error_type"].map({"FP": "#d95f02", "FN": "#1b9e77"}).tolist()
labels = [
    f"E{int(row.source_event_id)} {row.error_type}\n{row.label}"
    for row in error_plot.itertuples()
]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(labels, error_plot["y_probability"], color=colors)
ax.axvline(THRESHOLD, color="#333333", linestyle="--", linewidth=1, label="threshold 0.6")
ax.set_xlim(0, 1)
ax.set_xlabel("efd_possible probability")
ax.set_title("Expanded compact13 error probability ranking")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(PROBABILITY_PNG, bbox_inches="tight")
plt.close(fig)

heatmap_data = feature_profile.pivot_table(
    index="source_event_id",
    columns="feature_short",
    values="contribution",
    aggfunc="first",
).loc[sorted(EXPECTED_FP_EVENTS | EXPECTED_FN_EVENTS)]
fig, ax = plt.subplots(figsize=(13, 4.8))
max_abs = np.nanmax(np.abs(heatmap_data.to_numpy()))
im = ax.imshow(heatmap_data.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-max_abs, vmax=max_abs)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels([f"E{int(v)}" for v in heatmap_data.index])
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, rotation=70, ha="right", fontsize=7)
ax.set_title("Feature contribution heatmap for FP/FN events")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="contribution")
fig.tight_layout()
fig.savefig(FEATURE_HEATMAP_PNG, bbox_inches="tight")
plt.close(fig)

transition_counts = transition["transition_type"].value_counts().reindex(
    ["corrected_error", "new_error", "unchanged_error", "unchanged_correct"],
    fill_value=0,
)
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(transition_counts.index, transition_counts.values, color="#4c78a8")
ax.set_ylabel("event count")
ax.set_title("Reference to expanded prediction transition")
ax.tick_params(axis="x", rotation=25)
for idx, value in enumerate(transition_counts.values):
    ax.text(idx, value + 0.2, str(int(value)), ha="center", va="bottom")
fig.tight_layout()
fig.savefig(TRANSITION_SUMMARY_PNG, bbox_inches="tight")
plt.close(fig)

[PROBABILITY_PNG, FEATURE_HEATMAP_PNG, TRANSITION_SUMMARY_PNG]


[WindowsPath('07_데이터산출물/m1_expanded_compact13_error_probability_ranking.png'),
 WindowsPath('07_데이터산출물/m1_expanded_compact13_error_feature_heatmap.png'),
 WindowsPath('07_데이터산출물/m1_expanded_compact13_transition_summary.png')]

## 10. 보고서 생성


In [11]:
def markdown_table(df: pd.DataFrame, columns: list[str]) -> str:
    table = df[columns].copy()
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, row in table.iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in columns) + " |")
    return "\n".join([header, sep] + rows)


report_error = error_audit[audit_columns].sort_values(["error_type", "source_event_id"]).copy()
for col in ["y_probability", "reference_probability", "probability_delta_vs_reference", "coverage_rate"]:
    report_error[col] = report_error[col].map(lambda value: f"{float(value):.4f}")

report_transition_counts = transition_counts.reset_index()
report_transition_counts.columns = ["transition_type", "rows"]

report_ablation = ablation_metrics.loc[ablation_metrics["fold"].astype(str).eq("all")].copy()
for col in ["balanced_accuracy", "precision", "recall", "f1", "false_positive_rate"]:
    report_ablation[col] = report_ablation[col].map(lambda value: f"{float(value):.4f}")
report_ablation = report_ablation.sort_values("strategy")

report_decision = decision_df.copy()
for col in ["reference_value", "expanded_value", "delta"]:
    report_decision[col] = report_decision[col].map(lambda value: f"{float(value):.4f}")

report = f"""# M1 Expanded Compact13 오분류 Audit 보고서

## 개요

이번 단계는 `expanded_compact13 + threshold 0.6` 기준에서 남은 FP/FN 6건을 해석하고, expanded dataset을 다음 학습 기준으로 잠글 수 있는지 판단한 것이다.

최종 결론: **{final_decision}**

## 무엇을 했는지

- fixed eval 49행은 유지했다.
- FP Event 27, 28, 68과 FN Event 15, 40, 52를 event 단위로 audit했다.
- reference 대비 expanded 예측 변화가 새 오류인지, 기존 오류인지, 교정 오류인지 분류했다.
- weak positive와 candidate normal의 기여를 분리하기 위해 candidate ablation 4조건을 비교했다.
- compact13 coefficient contribution으로 각 오분류의 주요 drive feature를 정리했다.

## 오분류 6건

{markdown_table(report_error, ["error_type", "source_event_id", "substation_id", "label", "review_tag", "y_probability", "reference_probability", "probability_delta_vs_reference", "transition_type", "coverage_rate", "disturbance_count", "fault_window_overlap_count", "new_hard_normal_candidate", "label_quality_issue"])}

## Candidate Ablation

{markdown_table(report_ablation, ["strategy", "n", "balanced_accuracy", "precision", "recall", "f1", "false_positive_count", "false_negative_count", "false_positive_rate", "hard_normal_35_48_fp_count", "review_required_19_68_fp_count"])}

## Transition Summary

{markdown_table(report_transition_counts, ["transition_type", "rows"])}

## Final Dataset Lock Decision

{markdown_table(report_decision, ["criterion", "reference_value", "expanded_value", "delta", "pass", "final_decision"])}

## 변경 내용

| 항목 | 내용 |
| --- | --- |
| 노트북 | `06_노트북/14_m1_expanded_compact13_error_audit.ipynb` |
| 오분류 audit | `m1_expanded_compact13_error_audit.csv` |
| feature profile | `m1_expanded_compact13_error_feature_profile.csv` |
| transition audit | `m1_expanded_compact13_transition_audit.csv` |
| candidate ablation | `m1_expanded_compact13_candidate_ablation_metrics.csv` |
| 잠금 판단 | `m1_final_dataset_lock_decision.csv` |

## 검증

```mermaid
flowchart TD
  A[13 fixed eval predictions] --> B[FP/FN extraction]
  A --> C[reference vs expanded transition]
  D[feature pool] --> E[fold model replay]
  E --> F[feature contribution]
  D --> G[candidate ablation]
  B --> H[lock decision]
  C --> H
  F --> H
  G --> H
```

- fixed eval 49행을 유지했다.
- FP는 Event 27, 28, 68 총 3건으로 확인했다.
- FN은 Event 15, 40, 52 총 3건으로 확인했다.
- Event 20, 34, 69는 audit 대상에 포함되지 않았다.
- Event 19/68은 삭제하지 않고 review tag만 유지했다.
- candidate ablation 4조건 metric을 모두 생성했다.
- fold별 train/test substation overlap은 0이다.
- 학습 feature는 compact13 13개만 사용했다.

## 한계와 주의점

- 이번 단계는 final model 저장이 아니라 dataset 기준 잠금 여부 판단이다.
- Event 28은 새 hard normal 후보로 보되 normal 라벨을 바꾸지 않는다.
- FN Event 15/40/52는 모델 미탐으로 남지만 coverage와 fault label 품질 문제는 발견되지 않았다.

## 다음에 볼 것

- `expanded_compact13_dataset_lock` 기준으로 final training dataset 산출물을 만들 수 있다.
- 모델 저장은 다음 단계에서 별도로 수행한다.
"""

REPORT_PATH.write_text(report, encoding="utf-8")
print(REPORT_PATH)


07_데이터산출물\14_M1_expanded_compact13_error_audit_보고서.md


## 11. 검증


In [12]:
for path in [
    ERROR_AUDIT_PATH,
    ERROR_FEATURE_PROFILE_PATH,
    TRANSITION_AUDIT_PATH,
    CANDIDATE_ABLATION_METRICS_PATH,
    FINAL_LOCK_DECISION_PATH,
    PROBABILITY_PNG,
    FEATURE_HEATMAP_PNG,
    TRANSITION_SUMMARY_PNG,
    REPORT_PATH,
]:
    assert path.exists() and path.stat().st_size > 0, path

written_error_audit = pd.read_csv(ERROR_AUDIT_PATH)
written_transition = pd.read_csv(TRANSITION_AUDIT_PATH)
written_ablation = pd.read_csv(CANDIDATE_ABLATION_METRICS_PATH)
written_decision = pd.read_csv(FINAL_LOCK_DECISION_PATH)

assert len(fixed_eval) == 49
assert set(written_error_audit.loc[written_error_audit["error_type"].eq("FP"), "source_event_id"]) == EXPECTED_FP_EVENTS
assert set(written_error_audit.loc[written_error_audit["error_type"].eq("FN"), "source_event_id"]) == EXPECTED_FN_EVENTS
assert not written_error_audit["source_event_id"].isin(EXCLUDED_EVENT_IDS).any()
assert not written_transition["source_event_id"].isin(EXCLUDED_EVENT_IDS).any()
assert set(
    written_transition.loc[written_transition["source_event_id"].isin([19, 68]), "review_tag"].dropna()
) == {"review_required_normal"}
assert set(written_ablation.loc[written_ablation["fold"].astype(str).eq("all"), "strategy"]) == {
    "reference",
    "weak_only",
    "normal_only",
    "expanded_both",
}
assert len(COMPACT13_FEATURES) == 13
assert fold_group_audit["group_overlap_count"].eq(0).all()
assert written_decision["final_decision"].nunique() == 1
assert written_decision["final_decision"].iloc[0] in {
    "expanded_compact13_dataset_lock",
    "candidate_selection_review_required",
}

forbidden_terms = ["manufacturer" + "_2", "manufacturer " + "2", "M" + "2"]
check_paths = [
    NOTEBOOK_PATH,
    REPORT_PATH,
    ERROR_AUDIT_PATH,
    ERROR_FEATURE_PROFILE_PATH,
    TRANSITION_AUDIT_PATH,
    CANDIDATE_ABLATION_METRICS_PATH,
    FINAL_LOCK_DECISION_PATH,
]
hits = []
for path in check_paths:
    text = Path(path).read_text(encoding="utf-8", errors="ignore")
    if any(term in text for term in forbidden_terms):
        hits.append(str(path))
assert not hits, "비대상 제조사 문자열 발견"

validation_summary = {
    "fixed_eval_rows": len(fixed_eval),
    "fp_events": sorted(EXPECTED_FP_EVENTS),
    "fn_events": sorted(EXPECTED_FN_EVENTS),
    "ablation_strategies": sorted(written_ablation.loc[written_ablation["fold"].astype(str).eq("all"), "strategy"].tolist()),
    "final_decision": written_decision["final_decision"].iloc[0],
}
validation_summary


{'fixed_eval_rows': 49,
 'fp_events': [27, 28, 68],
 'fn_events': [15, 40, 52],
 'ablation_strategies': ['expanded_both',
  'normal_only',
  'reference',
  'weak_only'],
 'final_decision': 'expanded_compact13_dataset_lock'}